# Embeddings of a pretrained transformer with cosine similarity

Embeds both texts of every pair with a pretrained transformer WITHOUT fine-tuning and decides
by the cosine similarity of the two vectors whether they were written by the same author. One
forward pass per text gives two vectors - the first token (CLS) and the mean of all tokens -
and each gives its own result row. The threshold of each is selected on the validation pairs
from every similarity observed there and applied once to the test pairs.

A restarted run continues where it stopped - the similarities are saved in chunks, a finished
run is not repeated and chunks of other settings or other data stop the run.

In [ ]:
SMOKE = False  # True - a few rows of every data set, a chunk of 2 pairs
# label in the file names of a repeated run, so two runs never share a file
RUN_TAG = ""
# Hugging Face id, or the folder of a saved model
MODEL_ID = "allenai/longformer-base-4096"
DEVICE = "cuda"  # "cpu" only for the CPU checks - there is no fallback to the CPU

Standard libraries

In [ ]:
import functools
import json
import logging
import re
import shutil
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

Data and plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

Torch and Hugging Face

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel, AutoTokenizer, PreTrainedTokenizerBase

The shared modules of the thesis

In [ ]:
from avcore import data, evaluation, runs, tracking, transformer_training

# Configuration

The model - the notebook runs for these three models only, named after the end of MODEL_ID

In [ ]:
MODEL_NAMES = ["longformer-base-4096", "bigbird-roberta-base", "ModernBERT-large"]
MODEL_NAME = Path(MODEL_ID).name
if MODEL_NAME not in MODEL_NAMES:
    raise ValueError(f"MODEL_ID {MODEL_ID!r} does not end with one of {MODEL_NAMES}.")
POOLINGS = ["cls_embedding", "mean_pooling_embedding"]
CLASSIFIER = "cosine_similarity"
DEVELOPER_INITIALS = "JP"
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

Sizes - the token limit of a text (the same as the classification heads), the chunk size

In [ ]:
MAX_LENGTH = transformer_training.MAX_LENGTH
CHUNK_PAIRS = 500
SMOKE_ROWS = {"validation": 8, "test": 12}
if SMOKE:
    CHUNK_PAIRS = 2

The data files - the training pairs are not needed, nothing is trained

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
VALIDATION_PATH = DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE
TEST_PATH = DATA_DIR / "processed" / data.TEST_CLEANED_FILE
DATA_PATHS = [VALIDATION_PATH, TEST_PATH]

The run names - one per pooling, both part of every file name of their run

In [ ]:
def build_run_name(label: str) -> str:
    """Return the run name of a pooling, or of the chunk folder both poolings share.

    Args:
        label: The pooling, or "similarity_chunks" for the chunk folder.

    Returns:
        The name, with the smoke label and RUN_TAG when they are set.

    Raises:
        ValueError: If the name is not a valid file name.
    """
    parts = [DEVELOPER_INITIALS, MODEL_NAME, label]
    if SMOKE:
        parts.append("smoke")
    if RUN_TAG:
        parts.append(str(RUN_TAG))
    run_name = "_".join([*parts, "v2"])
    if not re.fullmatch(r"[\w.-]+", run_name):
        raise ValueError(f"Run name {run_name!r} is not a file name - change RUN_TAG.")
    return run_name

The result folders, one per pooling

In [ ]:
RESULTS_DIR = Path.home() / "results_v2"
if SMOKE:
    RESULTS_DIR = Path.home() / "results_v2_smoke" / str(RUN_TAG)
RUN_DIRS = {pooling: RESULTS_DIR / build_run_name(pooling) for pooling in POOLINGS}

Run context - the commit, the hashes of the lock file and of the data, the versions and the GPU.
It refuses a repository with uncommitted or unpushed changes.

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
device = torch.device(DEVICE)
# an unusable device stops the run here, before anything is read or written
torch.empty(0, device=device)
run_context = tracking.collect_run_context(PROJECT_DIR, DATA_PATHS)
print(f"Model {MODEL_NAME} on {device}, commit {run_context.environment['commit'][:7]}")

# Load data

In [ ]:
validation_df = data.read_jsonl(VALIDATION_PATH)
test_df = data.read_jsonl(TEST_PATH)
# a prediction table is keyed by the pair id - checked on the whole files
for name, pairs_df in {"validation": validation_df, "test": test_df}.items():
    if not pairs_df["id"].is_unique:
        raise ValueError(f"The {name} pairs have repeated ids.")
if SMOKE:
    validation_df = validation_df.head(SMOKE_ROWS["validation"])
    test_df = test_df.head(SMOKE_ROWS["test"])

# Set up model

The pretrained encoder without any head, in evaluation mode - no dropout, no gradients and no
random number (BigBird draws its random blocks only in training)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
pretrained_config = AutoConfig.from_pretrained(MODEL_ID)
# ModernBERT - no torch.compile, so the GPU runs the same code path as the CPU checks
is_modernbert = pretrained_config.model_type == "modernbert"
MODEL_LOAD_KWARGS = {"reference_compile": False} if is_modernbert else {}
model = AutoModel.from_pretrained(MODEL_ID, **MODEL_LOAD_KWARGS).to(device).eval()
# Longformer - global attention on the first token, as the classification head (a choice)
USES_GLOBAL_ATTENTION = pretrained_config.model_type == "longformer"
# BigBird - the configured attention, set again before every text
CONFIGURED_ATTENTION = getattr(pretrained_config, "attention_type", None)
compiled = getattr(model.config, "reference_compile", None)
print(f"attention {model.config._attn_implementation}, reference_compile {compiled}")

The settings of the run, the hashes of the two data files included - they have to match the
saved chunks and the result folders

In [ ]:
data_sha256 = [run_context.environment[f"data_sha256.{p.name}"] for p in DATA_PATHS]

Every setting that changes a similarity or a threshold

In [ ]:
run_settings = {
    "family": "transformer",
    "model": MODEL_NAME,
    "combination": "none",
    "classifier": CLASSIFIER,
    "model_id": MODEL_ID,
    "model_revision": pretrained_config._commit_hash,
    "model_load_kwargs": repr(MODEL_LOAD_KWARGS),
    "smoke": SMOKE,
    "max_length": MAX_LENGTH,
    "global_attention": "first token" if USES_GLOBAL_ATTENTION else "none",
    "validation_pairs": len(validation_df),
    "test_pairs": len(test_df),
    "threshold_candidates": "every validation similarity",
    "data_sha256": " ".join(data_sha256),
    "mlflow_experiment": EXPERIMENT_NAME,
}

A finished run is not repeated - every result folder has to hold these settings

In [ ]:
run_params = {
    pooling: {**run_settings, "representation": pooling} for pooling in POOLINGS
}
finished_runs = [
    runs.find_finished_run(RUN_DIRS[pooling], run_params[pooling])
    for pooling in POOLINGS
]
run_is_complete = all(finished_runs)

if run_is_complete:
    print("Both result folders exist - the run is already finished.")

The chunk folder outside the repository, one per run name as the checkpoints of the heads. It
holds the settings of its chunks, the commit and the GPU too (the numbers differ slightly between GPU
types); chunks of other settings, code or data stop the run

In [ ]:
CHUNK_DIR = Path.home() / "checkpoints_v2" / build_run_name("similarity_chunks")
CHUNK_SETTINGS_PATH = CHUNK_DIR / "settings.json"
# the commit too - chunks of other code are never resumed, also after a new tag
chunk_keys = ["commit", "gpu_name"]
chunk_settings = {**run_settings, **{k: run_context.environment[k] for k in chunk_keys}}
chunk_settings_text = json.dumps(chunk_settings, sort_keys=True, default=str)

A new chunk folder gets its settings first (a temporary file, then renamed)

In [ ]:
if not run_is_complete and not CHUNK_SETTINGS_PATH.is_file():
    CHUNK_DIR.mkdir(parents=True, exist_ok=True)
    partial_path = CHUNK_SETTINGS_PATH.with_suffix(".partial")
    partial_path.write_text(chunk_settings_text, encoding="utf-8")
    partial_path.rename(CHUNK_SETTINGS_PATH)

Chunks of other settings stop the run - they are deleted by hand

In [ ]:
if not run_is_complete:
    if CHUNK_SETTINGS_PATH.read_text(encoding="utf-8") != chunk_settings_text:
        raise RuntimeError(
            f"{CHUNK_DIR} holds the chunks of other settings - delete it."
        )

# Embed and compare

One forward pass per text, at most MAX_LENGTH tokens; both vectors from the last hidden state

The result of one text

In [ ]:
@dataclass
class TextEmbedding:
    """The vectors of one text and the attention it ran in."""

    vectors: dict[str, torch.Tensor]
    # BigBird only - the library switched the text to full attention (a short text)
    full_attention: bool

Embed one text

In [ ]:
def embed_text(
    text: str,
    *,
    model: torch.nn.Module,
    tokenizer: PreTrainedTokenizerBase,
    device: torch.device,
    uses_global_attention: bool,
    configured_attention: str | None,
) -> TextEmbedding:
    """Embed one text alone and return its vector of every pooling of POOLINGS.

    Args:
        text: The text.
        model: The pretrained encoder in evaluation mode.
        tokenizer: Its tokenizer.
        device: Device of the model.
        uses_global_attention: Longformer - global attention on the first token.
        configured_attention: BigBird - the attention type of its config, None otherwise.

    Returns:
        The CLS and the mean vector, whether the text ran in full attention and its
        number of tokens before the truncation.
    """
    if configured_attention is not None:
        # BigBird switches to full attention for good at a short text - every text starts
        # in the configured attention, so its vector does not depend on the texts before it
        model.set_attention_type(configured_attention)
    encoding = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    ).to(device)
    inputs = dict(encoding)
    if uses_global_attention:
        global_attention_mask = torch.zeros_like(encoding["input_ids"])
        global_attention_mask[:, 0] = 1
        inputs["global_attention_mask"] = global_attention_mask
    with torch.inference_mode():
        hidden_states = model(**inputs).last_hidden_state[0]
    # one text without padding - the mask is all ones, kept so batching stays correct
    mask = encoding["attention_mask"][0].unsqueeze(-1).to(hidden_states.dtype)
    mean_vector = (hidden_states * mask).sum(dim=0) / mask.sum()
    return TextEmbedding(
        vectors={
            "cls_embedding": hidden_states[0],
            "mean_pooling_embedding": mean_vector,
        },
        full_attention=configured_attention is not None
        and model.attention_type != configured_attention,
        n_tokens=len(tokenizer(text)["input_ids"]),
    )

The model and its settings bound once, so the chunks only pass the text

In [ ]:
embed = functools.partial(
    embed_text,
    model=model,
    tokenizer=tokenizer,
    device=device,
    uses_global_attention=USES_GLOBAL_ATTENTION,
    configured_attention=CONFIGURED_ATTENTION,
)

The similarities of the pairs of one chunk

In [ ]:
def compare_chunk(
    chunk_df: pd.DataFrame, embed: Callable[[str], TextEmbedding]
) -> pd.DataFrame:
    """Return the cosine similarity of both texts of every pair of a chunk, per pooling.

    Args:
        chunk_df: Pairs with the columns id, same and pair.
        embed: Embeds one text.

    Returns:
        One row per pair - id, same, one similarity column per pooling, the number of its
        texts in full attention, the tokens of both texts and its seconds.
    """
    rows = []
    for pair_id, same, texts in zip(
        chunk_df["id"], chunk_df["same"], chunk_df["pair"], strict=True
    ):
        start = time.perf_counter()
        first, second = (embed(text) for text in texts)
        row = {"id": pair_id, "same": int(same)}
        for pooling in POOLINGS:
            row[pooling] = F.cosine_similarity(
                first.vectors[pooling], second.vectors[pooling], dim=0
            ).item()
        row["full_attention_texts"] = int(first.full_attention) + int(
            second.full_attention
        )
        row["first_tokens"], row["second_tokens"] = first.n_tokens, second.n_tokens
        rows.append({**row, "seconds": time.perf_counter() - start})
    return pd.DataFrame(rows)

All similarities of a split - a chunk is saved once (written to a temporary file, then renamed)
and read back from its file, so a resumed run uses the same numbers as an uninterrupted one

In [ ]:
def compare_split(
    name: str,
    pairs_df: pd.DataFrame,
    *,
    embed: Callable[[str], TextEmbedding],
    chunk_dir: Path,
    chunk_pairs: int,
) -> pd.DataFrame:
    """Return the similarities of every pair of a split, computing only the missing chunks.

    Args:
        name: Name of the split, the prefix of its chunk files.
        pairs_df: The pairs of the split.
        embed: Embeds one text.
        chunk_dir: Folder of the chunk files.
        chunk_pairs: Pairs per chunk.

    Returns:
        The rows of compare_chunk for every pair, in the order of pairs_df.

    Raises:
        RuntimeError: If the chunks do not hold the pairs of the split in order.
    """
    chunk_dfs = []
    for start in range(0, len(pairs_df), chunk_pairs):
        chunk_path = chunk_dir / f"{name}-{start // chunk_pairs:05d}.csv"
        if not chunk_path.is_file():
            chunk_df = compare_chunk(pairs_df.iloc[start : start + chunk_pairs], embed)
            partial_path = chunk_path.with_suffix(".partial")
            chunk_df.to_csv(partial_path, index=False)
            partial_path.rename(chunk_path)
        chunk_dfs.append(
            pd.read_csv(chunk_path, dtype={"id": str}, float_precision="round_trip")
        )
    similarities_df = pd.concat(chunk_dfs, ignore_index=True)
    if similarities_df["id"].tolist() != pairs_df["id"].tolist():
        raise RuntimeError(f"The chunks of {name} do not hold its pairs in order.")
    return similarities_df

Compare the validation pairs, then the test pairs - the test pairs never select anything

In [ ]:
if not run_is_complete:
    validation_similarities_df = compare_split(
        "validation",
        validation_df,
        embed=embed,
        chunk_dir=CHUNK_DIR,
        chunk_pairs=CHUNK_PAIRS,
    )
    test_similarities_df = compare_split(
        "test", test_df, embed=embed, chunk_dir=CHUNK_DIR, chunk_pairs=CHUNK_PAIRS
    )

The confusion matrix of the test pairs

In [ ]:
def plot_confusion_matrix(predictions_df: pd.DataFrame, title: str) -> plt.Figure:
    """Plot the confusion matrix of a prediction table with counts.

    Args:
        predictions_df: Table with the columns same and pred.
        title: Title of the plot.

    Returns:
        The figure.
    """
    figure, axis = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        confusion_matrix(predictions_df["same"], predictions_df["pred"], labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axis,
        xticklabels=[False, True],
        yticklabels=[False, True],
    )
    axis.set(xlabel="Predicted", ylabel="Actual", title=title)
    return figure

The token counts over both texts of every pair of both splits (a text in several pairs counts once per pair),

In [ ]:
def count_tokens(scored_dfs: list[pd.DataFrame]) -> dict[str, int]:
    """Return the shortest and longest text and the texts cut to MAX_LENGTH.

    Args:
        scored_dfs: Outputs of compare_split, with the columns first_tokens and second_tokens.

    Returns:
        min_tokens, max_tokens and n_truncated_texts over both texts of every pair.
    """
    columns = ["first_tokens", "second_tokens"]
    tokens = pd.concat([df[column] for df in scored_dfs for column in columns])
    return {
        "min_tokens": int(tokens.min()),
        "max_tokens": int(tokens.max()),
        "n_truncated_texts": int((tokens > MAX_LENGTH).sum()),
    }

# Evaluate one pooling

The threshold of the highest validation F1 among every observed similarity (the lowest one if
two are equal), applied unchanged to the test pairs

The result of one pooling

In [ ]:
@dataclass
class PoolingResult:
    """Metrics, tables and the figure of one pooling, as written to its result folder."""

    metrics: dict[str, float]
    tables: dict[str, pd.DataFrame]
    figure: plt.Figure

Select the threshold and evaluate the test pairs

In [ ]:
def evaluate_pooling(
    pooling: str,
    *,
    validation_similarities_df: pd.DataFrame,
    test_similarities_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> PoolingResult:
    """Select the threshold of a pooling on the validation pairs and evaluate it on the test pairs.

    Args:
        pooling: One of POOLINGS.
        validation_similarities_df: Output of compare_split for the validation pairs.
        test_similarities_df: Output of compare_split for the test pairs.
        validation_df: The validation pairs.
        test_df: The test pairs.

    Returns:
        The metrics, the prediction and threshold tables and the test confusion matrix.
    """
    validation_scores_df = validation_similarities_df.rename(columns={pooling: "score"})
    test_scores_df = test_similarities_df.rename(columns={pooling: "score"})
    candidates = sorted(validation_scores_df["score"].unique())
    thresholds_df = evaluation.evaluate_classification_thresholds(
        validation_scores_df["same"], validation_scores_df["score"], candidates
    )
    threshold = float(evaluation.select_best_threshold(thresholds_df)["threshold"])
    predictions_df = runs.build_predictions(test_df, test_scores_df, threshold)
    test_metrics = evaluation.evaluate_results(
        predictions_df["same"], predictions_df["pred"]
    )
    print(f"{pooling}: threshold {threshold}, test {test_metrics}")
    scored_dfs = [validation_scores_df, test_scores_df]
    metrics = {
        "threshold": threshold,
        **{f"test_{key}": value for key, value in test_metrics.to_dict().items()},
        "n_threshold_candidates": len(candidates),
        # validation and test, the same in both rows of a run (one pass gives both)
        "n_full_attention_texts": sum(
            int(df["full_attention_texts"].sum()) for df in scored_dfs
        ),
        "scoring_seconds": sum(float(df["seconds"].sum()) for df in scored_dfs),
        **count_tokens(scored_dfs),
    }
    tables = {
        "predictions-validation": runs.build_predictions(
            validation_df, validation_scores_df, threshold
        ),
        "predictions-test": predictions_df,
        "thresholds-validation": thresholds_df,
    }
    title = f"Confusion Matrix: {MODEL_ID} - {pooling}"
    return PoolingResult(metrics, tables, plot_confusion_matrix(predictions_df, title))

# Save the results

Both poolings - a folder that already exists (a kill between the two) is not written again

In [ ]:
if not run_is_complete:
    for pooling in POOLINGS:
        if runs.find_finished_run(RUN_DIRS[pooling], run_params[pooling]):
            continue
        result = evaluate_pooling(
            pooling,
            validation_similarities_df=validation_similarities_df,
            test_similarities_df=test_similarities_df,
            validation_df=validation_df,
            test_df=test_df,
        )
        runs.remove_unfinished_run_dir(RUN_DIRS[pooling])
        tracking.write_run_files(
            RUN_DIRS[pooling],
            context=run_context,
            params=run_params[pooling],
            metrics=result.metrics,
            tables=result.tables,
            figures={"confusion_matrix-test": result.figure},
        )
        plt.close(result.figure)

The finished run needs no chunks any more

In [ ]:
if CHUNK_DIR.exists():
    shutil.rmtree(CHUNK_DIR)
for pooling in POOLINGS:
    print(RUN_DIRS[pooling])
    print((RUN_DIRS[pooling] / tracking.METRICS_FILE).read_text(encoding="utf-8"))